In [1]:
# Importations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb


In [2]:
# Chargement des données
df_full = pd.read_csv("donnees_finales_2nov.csv", low_memory=False)
print(f"Chargement de {len(df_full):,} lignes du CSV.")

target = 'conso_5_usages_ef'

df = df_full[df_full[target].notna()].copy()

n_sample = min(700_000, len(df))
df = df.sample(n=n_sample, random_state=42)
print(f"Échantillon de {n_sample:,} lignes prélevé.")

for i, col in enumerate(df.columns):
    print(f"{i}: {col}")


Chargement de 1,255,619 lignes du CSV.
Échantillon de 700,000 lignes prélevé.
0: conso_5_usages_ef
1: code_departement_ban
2: version_dpe
3: qualite_isolation_murs
4: type_batiment
5: classe_altitude
6: type_installation_ecs
7: type_energie_principale_chauffage
8: qualite_isolation_plancher_bas
9: adresse_ban
10: modele_dpe
11: zone_climatique
12: type_installation_chauffage
13: code_postal_ban
14: surface_habitable_logement
15: hauteur_sous_plafond
16: etiquette_dpe
17: nombre_niveau_logement
18: isolation_toiture
19: type_generateur_n1_ecs_n1
20: type_generateur_chauffage_principal
21: type_generateur_froid
22: _lat
23: _lon
24: _outlier
25: TX
26: TN
27: periode_construction


In [3]:
# Variables explicatives et préparation des données
variables_explicatives = [
    'qualite_isolation_murs',
    'type_batiment',
    'type_installation_ecs',
    'type_energie_principale_chauffage',
    'qualite_isolation_plancher_bas',
    'type_installation_chauffage',
    'surface_habitable_logement',
    'hauteur_sous_plafond',
    'nombre_niveau_logement',
    'isolation_toiture',
    'type_generateur_n1_ecs_n1',
    'type_generateur_chauffage_principal',
    'periode_construction',
    'code_postal_ban'
]

variables_explicatives = [v for v in variables_explicatives if v in df.columns]

for col in variables_explicatives:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

X = pd.get_dummies(df[variables_explicatives], drop_first=True)
y = np.log1p(df[target])

print(f"Données prêtes : {X.shape}")
print(f"Statistiques log de la cible : mean={y.mean():.2f}, std={y.std():.2f}")


Données prêtes : (700000, 66)
Statistiques log de la cible : mean=9.08, std=0.86


In [4]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Train : {X_train.shape[0]} lignes, Test : {X_test.shape[0]} lignes")


Train : 490000 lignes, Test : 210000 lignes


In [5]:
# Entraînement XGBoost Regressor (modèle optimisé)
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("Modèle XGBoost entraîné !")


Modèle XGBoost entraîné !


In [6]:
# Prédiction et métriques
y_pred = xgb_model.predict(X_test)

y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred)

mse = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_real, y_pred_real)

print(f"Mean Squared Error (MSE) : {mse:.2f}")
print(f"Root MSE (RMSE) : {rmse:.2f}")
print(f"Coefficient de détermination (R²) : {r2:.3f}")


Mean Squared Error (MSE) : 1027647107.16
Root MSE (RMSE) : 32056.94
Coefficient de détermination (R²) : 0.580


In [7]:
#Sauvegarde du modèle
joblib.dump(xgb_model, 'modele_conso_xgb_opt2.pkl', compress=3)
print(" Modèle XGBoost enregistré dans 'modele_conso_xgb_opt2.pkl'.")


 Modèle XGBoost enregistré dans 'modele_conso_xgb_opt2.pkl'.


In [59]:
# Optimisation des hyperparamètres XGBoost

from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

# Définir le modèle XGBoost de base
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_jobs=-1,
    random_state=42
)

# Grille de paramètres à tester
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [6, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 5, 10]
}

# Recherche aléatoire avec validation croisée 3-fold
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=20,  # nombre de combinaisons testées
    scoring='neg_root_mean_squared_error',  # RMSE inversé
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Entraînement
random_search.fit(X_train, y_train)

# Meilleurs paramètres
print(" Meilleurs hyperparamètres :")
print(random_search.best_params_)

# Évaluation sur le test set
best_model = random_search.best_estimator_
y_pred_opt = best_model.predict(X_test)

y_test_real = np.expm1(y_test)
y_pred_real_opt = np.expm1(y_pred_opt)

mse_opt = mean_squared_error(y_test_real, y_pred_real_opt)
rmse_opt = np.sqrt(mse_opt)
r2_opt = r2_score(y_test_real, y_pred_real_opt)

print(f"Optimisé - RMSE : {rmse_opt:.2f}, R² : {r2_opt:.3f}")


Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


✅ Meilleurs hyperparamètres :
{'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 15, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Optimisé - RMSE : 37012.08, R² : 0.527
